# NB02: LLM Style Benchmark

The main event. Tests whether LLMs replicate McCarthy's deep distributional structure.

4 task types x 8 models x 10-layer metric stack.

Depends on: NB00 (fingerprint), NB01 (contamination flags)

Cost estimate: $5-15

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from tqdm.notebook import tqdm

from lib.config import (
    CORPUS_PATH, FINGERPRINT_PATH, RESULTS_DIR,
    MODEL_ORDER, MODELS, SEED, MIN_WORDS_FOR_DELTA,
    MCCARTHY_SENTENCE_DIST, MCCARTHY_SYLLABLE_DIST,
)
from lib.api_clients import generate, generate_batch
from lib.mccarthy_metrics import (
    mccarthy_metrics, extract_words, pos_distribution,
    distinct_n, load_fingerprint, sentence_length_distribution,
    SENTENCE_BINS,
)
from lib.statistical import (
    bootstrap_ci, paired_bootstrap_test, jsd,
    jsd_from_counters, align_distributions,
)
from lib.prompts import (
    SYSTEM_PROMPT, SCENE_DESCRIPTIONS, TRANSFER_SOURCES,
    FEW_SHOT_EXAMPLES,
    build_continuation_prompt, build_zero_shot_prompt,
    build_few_shot_prompt, build_transfer_prompt,
)

sns.set_theme(style='whitegrid', font_scale=1.1)
np.random.seed(SEED)

BENCH_DIR = RESULTS_DIR / 'benchmark'
BENCH_DIR.mkdir(parents=True, exist_ok=True)

# Load fingerprint from NB00
fingerprint = load_fingerprint(FINGERPRINT_PATH)
print(f'Fingerprint loaded: {len(fingerprint)} keys')

# Load contamination flags from NB01
contam_path = RESULTS_DIR / 'contamination' / 'contamination_summary.csv'
if contam_path.exists():
    contam_df = pd.read_csv(contam_path, index_col=0)
    contaminated_models = contam_df[contam_df['Flagged Probes'] > 0].index.tolist()
    print(f'Contaminated models: {contaminated_models or "none"}')
else:
    contaminated_models = []
    print('Warning: NB01 not run yet. No contamination flags.')

## 1. Prepare Prompts

In [ ]:
# Load corpus for continuation passages
raw = CORPUS_PATH.read_text(encoding='utf-8')
corpus = '\n'.join(raw.split('\n')[55:])
corpus_words = corpus.split()

# Task 1: Continuation - 10 evenly spaced passages, 150-word prefix
PREFIX_WORDS = 150
N_CONTINUATIONS = 10
step = len(corpus_words) // (N_CONTINUATIONS + 1)

continuation_prompts = []
for i in range(N_CONTINUATIONS):
    start = step * (i + 1)
    prefix = ' '.join(corpus_words[start:start + PREFIX_WORDS])
    continuation_prompts.append(build_continuation_prompt(prefix))

# Task 2: Zero-shot scenes
zero_shot_prompts = [build_zero_shot_prompt(desc) for desc in SCENE_DESCRIPTIONS]

# Task 3: Few-shot scenes
few_shot_prompts = [build_few_shot_prompt(desc) for desc in SCENE_DESCRIPTIONS]

# Task 4: Style transfer
transfer_prompts = [build_transfer_prompt(p) for p in TRANSFER_SOURCES.values()]

all_tasks = {
    'continuation': continuation_prompts,
    'zero_shot': zero_shot_prompts,
    'few_shot': few_shot_prompts,
    'transfer': transfer_prompts,
}

total_calls = sum(len(p) for p in all_tasks.values()) * len(MODEL_ORDER)
print(f'Total API calls planned: {total_calls}')
for task, prompts in all_tasks.items():
    print(f'  {task}: {len(prompts)} prompts x {len(MODEL_ORDER)} models = {len(prompts) * len(MODEL_ORDER)} calls')

## 2. Generate All Responses

In [ ]:
# Generate responses for all models x all tasks
# Results stored as: generations[model_key][task_name] = [list of response strings]

generations = {}

for model_key in tqdm(MODEL_ORDER, desc='Models'):
    label = MODELS[model_key]['label']
    generations[model_key] = {}

    for task_name, prompts in all_tasks.items():
        print(f'  {label} / {task_name} ({len(prompts)} prompts)...')
        try:
            responses = generate_batch(
                model_key, prompts,
                system=SYSTEM_PROMPT,
                delay=1.5,
            )
            generations[model_key][task_name] = responses
        except Exception as e:
            print(f'    ERROR: {e}')
            generations[model_key][task_name] = []

# Save all generations
gen_path = BENCH_DIR / 'all_generations.json'
with open(gen_path, 'w') as f:
    json.dump(generations, f, indent=2)
print(f'\nSaved generations to {gen_path}')

## 3. Compute Metrics

Multi-layer metric stack applied to all generations.

In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm')

ref_pos = fingerprint.get('pos_unigram', {})
ref_pos_bi = fingerprint.get('pos_bigram', {})
# Convert pos_bigram keys back to tuples for JSD computation
ref_pos_bi_counter = Counter({tuple(k.split('->')): v for k, v in ref_pos_bi.items()})

ref_sent_dist = np.array([MCCARTHY_SENTENCE_DIST.get(b, 0) for b in SENTENCE_BINS]) / 100.0


def compute_all_metrics(text: str, model_key: str, task: str) -> dict:
    """Compute full metric stack for a single generation."""
    m = mccarthy_metrics(text)
    if 'error' in m:
        return {'model': model_key, 'task': task, 'error': m['error']}

    result = {
        'model': model_key,
        'model_label': MODELS[model_key]['label'],
        'task': task,
        'word_count': m['total_words'],
        # McCarthy-specific
        'mono_pct': m['monosyllable_pct'],
        'and_freq': m['and_frequency_pct'],
        'quotes': m['quotation_marks'],
        'avg_sent_len': m.get('avg_sentence_length', 0),
        # Diversity
        'distinct_1': distinct_n(text, 1),
        'distinct_2': distinct_n(text, 2),
    }

    # JSD on sentence length distribution
    gen_sent_dist = sentence_length_distribution(text)
    if gen_sent_dist.sum() > 0:
        result['jsd_sent_len'] = jsd(ref_sent_dist, gen_sent_dist)

    # POS bigram JSD
    pos = pos_distribution(text, nlp=nlp)
    gen_bi_counter = Counter(pos['bigram'])
    if gen_bi_counter:
        result['jsd_pos_bigram'] = jsd_from_counters(ref_pos_bi_counter, gen_bi_counter)

    return result

print('Metric computation function ready.')

In [ ]:
# Compute per-sample metrics
per_sample_metrics = []

for model_key in tqdm(MODEL_ORDER, desc='Computing metrics'):
    for task_name, responses in generations.get(model_key, {}).items():
        for response in responses:
            if response:
                m = compute_all_metrics(response, model_key, task_name)
                per_sample_metrics.append(m)

df_samples = pd.DataFrame(per_sample_metrics)
print(f'Computed metrics for {len(df_samples)} samples')
df_samples.head()

## 4. Aggregate Metrics (Concatenated for Delta)

Per Eder 2015: individual 300-word samples are too short for reliable stylometry.
Concatenate all generations per model to 5000+ words before computing Burrows' Delta.

In [ ]:
from faststylometry import load_corpus_from_strings, calculate_burrows_delta

# Build concatenated texts per model
model_texts = {}
for model_key in MODEL_ORDER:
    all_responses = []
    for task_name, responses in generations.get(model_key, {}).items():
        all_responses.extend([r for r in responses if r])
    concat = ' '.join(all_responses)
    word_count = len(concat.split())
    model_texts[model_key] = concat
    label = MODELS[model_key]['label']
    flag = '*' if label in contaminated_models else ''
    print(f'{label}{flag}: {word_count:,} words (min {MIN_WORDS_FOR_DELTA:,} for Delta)')

# Build faststylometry corpus: McCarthy reference + each model
# Reference = full Blood Meridian
ref_corpus = load_corpus_from_strings(
    authors=['McCarthy'],
    titles=['blood_meridian'],
    texts=[corpus],
)

# Each model as a separate "author"
gen_authors = [MODELS[k]['label'] for k in MODEL_ORDER]
gen_titles = [k for k in MODEL_ORDER]
gen_texts = [model_texts[k] for k in MODEL_ORDER]

gen_corpus = load_corpus_from_strings(
    authors=gen_authors,
    titles=gen_titles,
    texts=gen_texts,
)

# Compute Delta
delta_results = calculate_burrows_delta(ref_corpus, gen_corpus)
print('\nBurrows\' Delta scores (lower = closer to McCarthy):')
print(delta_results)

## 5. MAUVE Scores

In [ ]:
import mauve

# MAUVE needs reference text and generated text
# Split Blood Meridian into comparable-length chunks as reference
ref_words = corpus.split()
chunk_size = 300
ref_chunks = [' '.join(ref_words[i:i+chunk_size])
              for i in range(0, len(ref_words), chunk_size)
              if len(ref_words[i:i+chunk_size]) >= chunk_size // 2]

mauve_scores = {}
for model_key in tqdm(MODEL_ORDER, desc='MAUVE'):
    gen_chunks = []
    for task_name, responses in generations.get(model_key, {}).items():
        gen_chunks.extend([r for r in responses if r and len(r.split()) > 50])

    if len(gen_chunks) < 5:
        print(f'{MODELS[model_key]["label"]}: too few samples for MAUVE')
        continue

    # Use subset of reference to match generated count
    n = min(len(gen_chunks), len(ref_chunks))
    result = mauve.compute_mauve(
        p_text=ref_chunks[:n],
        q_text=gen_chunks[:n],
        device_id=-1,  # CPU
        max_text_length=512,
        verbose=False,
    )
    mauve_scores[model_key] = result.mauve
    print(f'{MODELS[model_key]["label"]}: MAUVE = {result.mauve:.4f}')

print('\nMAUVE scores (higher = closer to McCarthy distribution):')
for k in MODEL_ORDER:
    if k in mauve_scores:
        print(f'  {MODELS[k]["label"]:20s} {mauve_scores[k]:.4f}')

## 6. BERTScore (Continuation Task Only)

In [ ]:
from bert_score import score as bert_score_fn

# BERTScore only makes sense for continuation where we have reference text
# Reference = actual continuation from corpus
continuation_refs = []
for i in range(N_CONTINUATIONS):
    start = step * (i + 1) + PREFIX_WORDS
    ref_text = ' '.join(corpus_words[start:start + 300])
    continuation_refs.append(ref_text)

bertscore_results = {}
for model_key in tqdm(MODEL_ORDER, desc='BERTScore'):
    cont_gens = generations.get(model_key, {}).get('continuation', [])
    if not cont_gens:
        continue

    n = min(len(cont_gens), len(continuation_refs))
    P, R, F1 = bert_score_fn(
        cont_gens[:n], continuation_refs[:n],
        lang='en', verbose=False,
    )
    bertscore_results[model_key] = {
        'precision': P.mean().item(),
        'recall': R.mean().item(),
        'f1': F1.mean().item(),
    }
    print(f'{MODELS[model_key]["label"]}: BERTScore F1 = {F1.mean():.4f}')

## 7. Bootstrap Confidence Intervals

In [ ]:
# Bootstrap CIs for key per-sample metrics
ci_metrics = ['mono_pct', 'and_freq', 'avg_sent_len', 'distinct_1', 'distinct_2']

ci_results = []
for model_key in MODEL_ORDER:
    label = MODELS[model_key]['label']
    model_df = df_samples[df_samples['model'] == model_key]

    for metric in ci_metrics:
        values = model_df[metric].dropna().values
        if len(values) < 3:
            continue
        point, lower, upper = bootstrap_ci(values)
        ci_results.append({
            'model': label,
            'metric': metric,
            'point': point,
            'ci_lower': lower,
            'ci_upper': upper,
        })

df_ci = pd.DataFrame(ci_results)
print(df_ci.pivot(index='model', columns='metric', values='point').round(2))

## 8. LLM-as-Judge (Pairwise Elo)

In [ ]:
from lib.prompts import build_judge_prompt
from itertools import combinations

# Use Claude as judge (no CoT per LitBench 2025)
JUDGE_MODEL = 'claude-haiku'  # cheaper for judging

# Pick one representative generation per model (zero-shot, scene 0)
judge_samples = {}
for model_key in MODEL_ORDER:
    zs = generations.get(model_key, {}).get('zero_shot', [])
    if zs:
        judge_samples[model_key] = zs[0]

# Pairwise comparisons
wins = Counter()
model_keys_with_samples = [k for k in MODEL_ORDER if k in judge_samples]

for a, b in tqdm(list(combinations(model_keys_with_samples, 2)), desc='Judging'):
    prompt = build_judge_prompt(judge_samples[a], judge_samples[b])
    try:
        verdict = generate(JUDGE_MODEL, prompt, temperature=0.0, max_tokens=5)
        verdict = verdict.strip().upper()
        if 'A' in verdict:
            wins[a] += 1
        elif 'B' in verdict:
            wins[b] += 1
    except Exception as e:
        print(f'Judge error ({a} vs {b}): {e}')

# Simple Elo from win counts
total_comparisons = len(list(combinations(model_keys_with_samples, 2)))
print('\nPairwise wins:')
for k in MODEL_ORDER:
    if k in wins:
        print(f'  {MODELS[k]["label"]:20s} {wins[k]} wins')

## 9. Results Summary

In [ ]:
# Build master results table
summary_rows = []
for model_key in MODEL_ORDER:
    label = MODELS[model_key]['label']
    flag = '*' if label in contaminated_models else ''
    model_df = df_samples[df_samples['model'] == model_key]

    row = {
        'Model': f'{label}{flag}',
        'Mono%': model_df['mono_pct'].mean(),
        'And%': model_df['and_freq'].mean(),
        'AvgSentLen': model_df['avg_sent_len'].mean(),
        'Distinct-1': model_df['distinct_1'].mean(),
        'Distinct-2': model_df['distinct_2'].mean(),
        'JSD(SentLen)': model_df['jsd_sent_len'].mean() if 'jsd_sent_len' in model_df else None,
        'JSD(POS)': model_df['jsd_pos_bigram'].mean() if 'jsd_pos_bigram' in model_df else None,
        'MAUVE': mauve_scores.get(model_key),
        'BERTScore': bertscore_results.get(model_key, {}).get('f1'),
        'Judge Wins': wins.get(model_key, 0),
    }
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)

# Add McCarthy baseline row
baseline = {
    'Model': 'McCarthy (baseline)',
    'Mono%': 81.6,
    'And%': 5.82,
    'AvgSentLen': 15.7,
    'Distinct-1': None,
    'Distinct-2': None,
    'JSD(SentLen)': 0.0,
    'JSD(POS)': 0.0,
    'MAUVE': 1.0,
    'BERTScore': None,
    'Judge Wins': None,
}
df_summary = pd.concat([pd.DataFrame([baseline]), df_summary], ignore_index=True)

print('=== MASTER RESULTS TABLE ===')
print(df_summary.round(3).to_string(index=False))

# Save
df_summary.to_csv(BENCH_DIR / 'benchmark_summary.csv', index=False)
df_samples.to_csv(BENCH_DIR / 'per_sample_metrics.csv', index=False)

## 10. Visualization: Radar Charts + Heatmap

In [ ]:
# Radar chart: each model's deviation from McCarthy baseline
radar_metrics = ['Mono%', 'And%', 'AvgSentLen', 'Distinct-1', 'MAUVE']
# Normalize each metric to [0, 1] range for radar
# For metrics where closer to McCarthy = better, compute distance

fig, axes = plt.subplots(2, 4, figsize=(20, 10), subplot_kw=dict(polar=True))
axes = axes.flatten()

categories = ['Mono%\n(81.6)', 'And%\n(5.82)', 'SentLen\n(15.7)',
              'Distinct-1', 'MAUVE']
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]  # close the loop

for idx, model_key in enumerate(MODEL_ORDER):
    row = df_summary[df_summary['Model'].str.contains(MODELS[model_key]['label'])]
    if row.empty:
        continue
    row = row.iloc[0]

    # Normalize: how close is each metric to McCarthy?
    values = [
        1 - abs(row['Mono%'] - 81.6) / 81.6 if pd.notna(row['Mono%']) else 0,
        1 - abs(row['And%'] - 5.82) / 5.82 if pd.notna(row['And%']) else 0,
        1 - abs(row['AvgSentLen'] - 15.7) / 15.7 if pd.notna(row['AvgSentLen']) else 0,
        row['Distinct-1'] if pd.notna(row['Distinct-1']) else 0,
        row['MAUVE'] if pd.notna(row['MAUVE']) else 0,
    ]
    values = [max(0, min(1, v)) for v in values]  # clamp to [0, 1]
    values += values[:1]

    ax = axes[idx]
    ax.plot(angles, values, 'o-', linewidth=2)
    ax.fill(angles, values, alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, size=8)
    ax.set_ylim(0, 1)
    ax.set_title(MODELS[model_key]['label'], size=11, y=1.1)

plt.tight_layout()
plt.savefig(BENCH_DIR / 'radar_charts.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap: models x metrics
heat_metrics = ['Mono%', 'And%', 'AvgSentLen', 'JSD(SentLen)', 'JSD(POS)',
                'Distinct-1', 'Distinct-2', 'MAUVE']
heat_data = df_summary[df_summary['Model'] != 'McCarthy (baseline)'][['Model'] + heat_metrics].set_index('Model')

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(heat_data.astype(float), annot=True, fmt='.2f', cmap='RdYlGn',
            ax=ax, linewidths=0.5)
ax.set_title('LLM Style Benchmark: Models x Metrics')
plt.tight_layout()
plt.savefig(BENCH_DIR / 'heatmap.png', dpi=150, bbox_inches='tight')
plt.show()